# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alien-is-here/FlyRank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Answer:**

I chose Random Forest as the modeling method because it fits the capstone modeling lane and can capture nonlinear relationships between the selected features and the target. It does not require feature scaling and can handle numerical features with different ranges. It also provides feature importance, which helps with model interpretation.

I use Random Forest as a stronger model to compare against the simpler Week-4 baseline and to evaluate whether the selected performance signals provide useful information for identifying observed declining pages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
feature_cols = [
    "ctr",
    "avg_position",
    "engagement_rate"
]

X = df[feature_cols]

y = (df["trend_direction"] == "down").astype(int)

groups = df[["client_id","content_id"]]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [9]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

X_train: (24000, 3)
X_test: (6000, 3)

y_train distribution:
trend_direction
1    13010
0    10990
Name: count, dtype: int64

y_test distribution:
trend_direction
1    3252
0    2748
Name: count, dtype: int64


I use an 80/20 train-test split with stratification. This keeps 80% of the data for training and 20% for testing, while stratify=y preserves the proportion of declining and non-declining pages in both sets. The test data remains unseen during training so the model can be evaluated on held-out examples.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

# Predict on the same test set
y_pred = rf_model.predict(X_test)

# Random Forest metrics
rf_precision = precision_score(y_test, y_pred)
rf_recall = recall_score(y_test, y_pred)
rf_f1 = f1_score(y_test, y_pred)

print("Random Forest")
print("Precision:", rf_precision)
print("Recall:", rf_recall)
print("F1:", rf_f1)

Random Forest
Precision: 0.6083382266588373
Recall: 0.6371463714637147
F1: 0.6224091318714329


In [11]:
baseline_test = df.loc[X_test.index].copy()

baseline_test["baseline_score"] = (
    baseline_test["days_since_last_update"].astype(float)
    * baseline_test["impressions_90d"].astype(float)
    * baseline_test["trend_direction"].eq("down").map({
        True: 1.0,
        False: 0.5
    })
)

# Rank by baseline score
baseline_test = baseline_test.sort_values(
    "baseline_score",
    ascending=False
)

# Select top 20% as positive predictions
cutoff = int(len(baseline_test) * 0.20)

baseline_test["baseline_pred"] = 0
baseline_test.iloc[
    :cutoff,
    baseline_test.columns.get_loc("baseline_pred")
] = 1

# Return to original X_test order
baseline_test = baseline_test.loc[X_test.index]

y_baseline = baseline_test["baseline_pred"]

# Baseline metrics
baseline_precision = precision_score(y_test, y_baseline)
baseline_recall = recall_score(y_test, y_baseline)
baseline_f1 = f1_score(y_test, y_baseline)

In [12]:
comparison = pd.DataFrame({
    "Model": ["Week-4 Baseline", "Random Forest"],
    "Precision": [baseline_precision, rf_precision],
    "Recall": [baseline_recall, rf_recall],
    "F1": [baseline_f1, rf_f1]
})

comparison

,Model,Precision,Recall,F1
0,Week-4 Baseline,0.650000,0.239852,0.350404
1,Random Forest,0.608338,0.637146,0.622409


The Week-4 baseline has slightly higher precision, but the Random Forest has substantially higher recall and F1. Therefore, the Random Forest provides a better overall balance for identifying observed declining pages in this experiment.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
# Error analysis
error_df = X_test.copy()
error_df["actual"] = y_test
error_df["predicted"] = y_pred

false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
]

false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 1334
False negatives: 1180


In [14]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df

,feature,importance
1,avg_position,0.595183
0,ctr,0.245252
2,engagement_rate,0.159565


The Random Forest produced 1,334 false positives and 1,180 false negatives, so it still makes both types of errors when identifying declining pages. The model relied most on avg_position (0.595), followed by ctr (0.245) and engagement_rate (0.160). This shows which features the model relied on most, but feature importance does not mean that these features cause decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.